In [10]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from rich import print

# An in-memory checkpoint saver.
# This checkpoint saver stores checkpoints in memory using a defaultdict.
# Note:
# Only use InMemorySaver for debugging or testing purposes.
# For production use cases we recommend installing langgraph-checkpoint-postgres and using PostgresSaver / AsyncPostgresSaver.
# If you are using LangSmith Deployment, no checkpointer needs to be specified. The correct managed checkpointer will be used automatically.
# Examples
#     import asyncio

#     from langgraph.checkpoint.memory import InMemorySaver
#     from langgraph.graph import StateGraph

#     builder = StateGraph(int)
#     builder.add_node("add_one", lambda x: x + 1)
#     builder.set_entry_point("add_one")
#     builder.set_finish_point("add_one")

#     memory = InMemorySaver()
#     graph = builder.compile(checkpointer=memory)
#     coro = graph.ainvoke(1, {"configurable": {"thread_id": "thread-1"}})
#     asyncio.run(coro)  # Output: 2

In [4]:
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')

In [5]:
class jokestate(TypedDict):

    topic : str
    joke : str
    explanation : str

In [6]:
def generate_joke(state:jokestate):

    topic = state['topic']

    prompt = f"Generate best funny joke on this {topic}"

    response = llm.invoke(prompt)

    return {'joke':response}

In [7]:
def generate_explanation(state:jokestate):

    generated_joke = state['joke']

    explanation_prompt = f"Give me 2 lines explanation for this generated joke{generated_joke}"

    response = llm.invoke(explanation_prompt)

    return {'explanation':response}

In [8]:
graph = StateGraph(jokestate)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

memory_saver = InMemorySaver()
joke_explanatin = graph.compile(checkpointer = memory_saver)



In [12]:
config1 = {'configurable':{'thread_id':'1'}}
print(joke_explanatin.invoke({'topic':'cricket'},config=config1))

{
    'topic': 'cricket',
    'joke': AIMessage(
        content="Here are a few funny cricket jokes, pick the one you like best!\n\n**Joke 1 (Playing on 
words):**\n\n> Why did the cricket ball break up with the bat?\n>\n> Because it felt like it was always being **hit
on**!\n\n**Joke 2 (Relatable frustration):**\n\n> My cricket team is so bad, our coach told us to practice our 
batting.\n>\n> We spent two hours practicing how to **duck**!\n\n**Joke 3 (Slightly absurd):**\n\n> What's the 
difference between a cricket fan and a pigeon?\n>\n> The pigeon can still make a deposit on a car.\n\n**Joke 4 
(Short and sweet):**\n\n> Why are cricketers such bad dancers?\n>\n> Because they're always afraid of getting 
**caught**!\n\n**Joke 5 (A bit of a dad joke):**\n\n> I tried to tell a joke about cricket, but it went for a 
**six**.\n\nLet me know if you'd like another one!",
        additional_kwargs={},
        response_metadata={
            'finish_reason': 'STOP',
            'model_name': 'gemini-2.5-flash-lite',
            'safety_ratings': [],
            'model_provider': 'google_genai'
        },
        id='lc_run--019e0360-a70f-7451-aedc-9022dc77bd9f-0',
        tool_calls=[],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 8,
            'output_tokens': 223,
            'total_tokens': 231,
            'input_token_details': {'cache_read': 0}
        }
    ),
    'explanation': AIMessage(
        content='This joke content presents a collection of five cricket-themed jokes. The jokes utilize wordplay, 
relatable scenarios, and lighthearted absurdity to entertain.',
        additional_kwargs={},
        response_metadata={
            'finish_reason': 'STOP',
            'model_name': 'gemini-2.5-flash-lite',
            'safety_ratings': [],
            'model_provider': 'google_genai'
        },
        id='lc_run--019e0360-b8bd-7a60-a8a9-d9471a9cb69d-0',
        tool_calls=[],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 411,
            'output_tokens': 29,
            'total_tokens': 440,
            'input_token_details': {'cache_read': 0}
        }
    )
}

In [ ]:
print(joke_explanatin.get_state(config1))

StateSnapshot(
    values={
        'topic': 'house',
        'joke': AIMessage(
            content='Okay, let\'s get some laughs out of this house! To give you the *best* funny joke, I need a 
little more information. Tell me about the house!\n\n**For example, is it:**\n\n*   **Old and creaky?**\n*   
**Brand new and minimalist?**\n*   **Messy and chaotic?**\n*   **Unusually shaped?**\n*   **Full of quirky 
features?**\n*   **Painted a ridiculous color?**\n*   **Having any specific problems (leaky roof, strange noises, 
etc.)?**\n*   **Just generally... *a house*?**\n\n**In the meantime, here are a few generic funny jokes that 
*could* apply to a house, depending on the vibe:**\n\n**If it\'s a bit run-down:**\n\n> I asked the house if it 
needed any repairs. It just sighed and said, "Honey, I need a whole new life."\n\n**If it\'s too small:**\n\n> This
house is so small, when the cat sneezes, we all have to go outside for fresh air.\n\n**If it\'s a bit weird:**\n\n>
I\'m pretty sure this house has a secret passage. Or maybe that\'s just where the spiders are hiding their secret 
society meetings.\n\n**If it\'s just... a house:**\n\n> Why did the house go to therapy? Because it had too many 
*issues* with its foundation!\n\n**To give you the *perfect* joke, tell me something specific about the house! I\'m
ready to make it funny!**',
            additional_kwargs={},
            response_metadata={
                'finish_reason': 'STOP',
                'model_name': 'gemini-2.5-flash-lite',
                'safety_ratings': [],
                'model_provider': 'google_genai'
            },
            id='lc_run--019e0363-f5dd-7ca2-92ec-c6e2008565f8-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 8,
                'output_tokens': 339,
                'total_tokens': 347,
                'input_token_details': {'cache_read': 0}
            }
        ),
        'explanation': AIMessage(
            content='The joke generator asks for details about the house to craft a specific, funny joke. It 
provides examples and generic jokes while waiting for more information.',
            additional_kwargs={},
            response_metadata={
                'finish_reason': 'STOP',
                'model_name': 'gemini-2.5-flash-lite',
                'safety_ratings': [],
                'model_provider': 'google_genai'
            },
            id='lc_run--019e0364-1044-7303-876d-d8aba1ffb08b-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 539,
                'output_tokens': 29,
                'total_tokens': 568,
                'input_token_details': {'cache_read': 0}
            }
        )
    },
    next=(),
    config={
        'configurable': {
            'thread_id': '2',
            'checkpoint_ns': '',
            'checkpoint_id': '1f14a368-b39a-620d-8002-9502050268ba'
        }
    },
    metadata={'source': 'loop', 'step': 2, 'parents': {}},
    created_at='2026-05-07T17:02:34.969537+00:00',
    parent_config={
        'configurable': {
            'thread_id': '2',
            'checkpoint_ns': '',
            'checkpoint_id': '1f14a368-83c7-6e28-8001-9d6d40540bb0'
        }
    },
    tasks=(),
    interrupts=()
)

In [17]:
print(list(joke_explanatin.get_state_history(config1)))

[
    StateSnapshot(
        values={
            'topic': 'cricket',
            'joke': AIMessage(
                content="Here are a few funny cricket jokes, pick the one you like best!\n\n**Joke 1 (Playing on 
words):**\n\n> Why did the cricket ball break up with the bat?\n>\n> Because it felt like it was always being **hit
on**!\n\n**Joke 2 (Relatable frustration):**\n\n> My cricket team is so bad, our coach told us to practice our 
batting.\n>\n> We spent two hours practicing how to **duck**!\n\n**Joke 3 (Slightly absurd):**\n\n> What's the 
difference between a cricket fan and a pigeon?\n>\n> The pigeon can still make a deposit on a car.\n\n**Joke 4 
(Short and sweet):**\n\n> Why are cricketers such bad dancers?\n>\n> Because they're always afraid of getting 
**caught**!\n\n**Joke 5 (A bit of a dad joke):**\n\n> I tried to tell a joke about cricket, but it went for a 
**six**.\n\nLet me know if you'd like another one!",
                additional_kwargs={},
                response_metadata={
                    'finish_reason': 'STOP',
                    'model_name': 'gemini-2.5-flash-lite',
                    'safety_ratings': [],
                    'model_provider': 'google_genai'
                },
                id='lc_run--019e0360-a70f-7451-aedc-9022dc77bd9f-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 8,
                    'output_tokens': 223,
                    'total_tokens': 231,
                    'input_token_details': {'cache_read': 0}
                }
            ),
            'explanation': AIMessage(
                content='This joke content presents a collection of five cricket-themed jokes. The jokes utilize 
wordplay, relatable scenarios, and lighthearted absurdity to entertain.',
                additional_kwargs={},
                response_metadata={
                    'finish_reason': 'STOP',
                    'model_name': 'gemini-2.5-flash-lite',
                    'safety_ratings': [],
                    'model_provider': 'google_genai'
                },
                id='lc_run--019e0360-b8bd-7a60-a8a9-d9471a9cb69d-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 411,
                    'output_tokens': 29,
                    'total_tokens': 440,
                    'input_token_details': {'cache_read': 0}
                }
            )
        },
        next=(),
        config={
            'configurable': {
                'thread_id': '1',
                'checkpoint_ns': '',
                'checkpoint_id': '1f14a360-8bab-618c-800a-b8ca2f156f10'
            }
        },
        metadata={'source': 'loop', 'step': 10, 'parents': {}},
        created_at='2026-05-07T16:58:56.033806+00:00',
        parent_config={
            'configurable': {
                'thread_id': '1',
                'checkpoint_ns': '',
                'checkpoint_id': '1f14a360-5b16-6f14-8009-8babde9132b3'
            }
        },
        tasks=(),
        interrupts=()
    ),
    StateSnapshot(
        values={
            'topic': 'cricket',
            'joke': AIMessage(
                content="Here are a few funny cricket jokes, pick the one you like best!\n\n**Joke 1 (Playing on 
words):**\n\n> Why did the cricket ball break up with the bat?\n>\n> Because it felt like it was always being **hit
on**!\n\n**Joke 2 (Relatable frustration):**\n\n> My cricket team is so bad, our coach told us to practice our 
batting.\n>\n> We spent two hours practicing how to **duck**!\n\n**Joke 3 (Slightly absurd):**\n\n> What's the 
difference between a cricket fan and a pigeon?\n>\n> The pigeon can still make a deposit on a car.\n\n**Joke 4 
(Short and sweet):**\n\n> Why are cricketers such bad dancers?\n>\n> Because they're always afraid of getting 
**caught**!\n\n**Joke 5 (A bit of a dad joke):**\n\n> I tried to 

In [18]:
config2 = {'configurable':{'thread_id':'2'}}
print(joke_explanatin.invoke({'topic':'house'},config=config2))

{
    'topic': 'house',
    'joke': AIMessage(
        content='Okay, let\'s get some laughs out of this house! To give you the *best* funny joke, I need a little
more information. Tell me about the house!\n\n**For example, is it:**\n\n*   **Old and creaky?**\n*   **Brand new 
and minimalist?**\n*   **Messy and chaotic?**\n*   **Unusually shaped?**\n*   **Full of quirky features?**\n*   
**Painted a ridiculous color?**\n*   **Having any specific problems (leaky roof, strange noises, etc.)?**\n*   
**Just generally... *a house*?**\n\n**In the meantime, here are a few generic funny jokes that *could* apply to a 
house, depending on the vibe:**\n\n**If it\'s a bit run-down:**\n\n> I asked the house if it needed any repairs. It
just sighed and said, "Honey, I need a whole new life."\n\n**If it\'s too small:**\n\n> This house is so small, 
when the cat sneezes, we all have to go outside for fresh air.\n\n**If it\'s a bit weird:**\n\n> I\'m pretty sure 
this house has a secret passage. Or maybe that\'s just where the spiders are hiding their secret society 
meetings.\n\n**If it\'s just... a house:**\n\n> Why did the house go to therapy? Because it had too many *issues* 
with its foundation!\n\n**To give you the *perfect* joke, tell me something specific about the house! I\'m ready to
make it funny!**',
        additional_kwargs={},
        response_metadata={
            'finish_reason': 'STOP',
            'model_name': 'gemini-2.5-flash-lite',
            'safety_ratings': [],
            'model_provider': 'google_genai'
        },
        id='lc_run--019e0363-f5dd-7ca2-92ec-c6e2008565f8-0',
        tool_calls=[],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 8,
            'output_tokens': 339,
            'total_tokens': 347,
            'input_token_details': {'cache_read': 0}
        }
    ),
    'explanation': AIMessage(
        content='The joke generator asks for details about the house to craft a specific, funny joke. It provides 
examples and generic jokes while waiting for more information.',
        additional_kwargs={},
        response_metadata={
            'finish_reason': 'STOP',
            'model_name': 'gemini-2.5-flash-lite',
            'safety_ratings': [],
            'model_provider': 'google_genai'
        },
        id='lc_run--019e0364-1044-7303-876d-d8aba1ffb08b-0',
        tool_calls=[],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 539,
            'output_tokens': 29,
            'total_tokens': 568,
            'input_token_details': {'cache_read': 0}
        }
    )
}

In [20]:
print(list(joke_explanatin.get_state(config2)))

[
    {
        'topic': 'house',
        'joke': AIMessage(
            content='Okay, let\'s get some laughs out of this house! To give you the *best* funny joke, I need a 
little more information. Tell me about the house!\n\n**For example, is it:**\n\n*   **Old and creaky?**\n*   
**Brand new and minimalist?**\n*   **Messy and chaotic?**\n*   **Unusually shaped?**\n*   **Full of quirky 
features?**\n*   **Painted a ridiculous color?**\n*   **Having any specific problems (leaky roof, strange noises, 
etc.)?**\n*   **Just generally... *a house*?**\n\n**In the meantime, here are a few generic funny jokes that 
*could* apply to a house, depending on the vibe:**\n\n**If it\'s a bit run-down:**\n\n> I asked the house if it 
needed any repairs. It just sighed and said, "Honey, I need a whole new life."\n\n**If it\'s too small:**\n\n> This
house is so small, when the cat sneezes, we all have to go outside for fresh air.\n\n**If it\'s a bit weird:**\n\n>
I\'m pretty sure this house has a secret passage. Or maybe that\'s just where the spiders are hiding their secret 
society meetings.\n\n**If it\'s just... a house:**\n\n> Why did the house go to therapy? Because it had too many 
*issues* with its foundation!\n\n**To give you the *perfect* joke, tell me something specific about the house! I\'m
ready to make it funny!**',
            additional_kwargs={},
            response_metadata={
                'finish_reason': 'STOP',
                'model_name': 'gemini-2.5-flash-lite',
                'safety_ratings': [],
                'model_provider': 'google_genai'
            },
            id='lc_run--019e0363-f5dd-7ca2-92ec-c6e2008565f8-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 8,
                'output_tokens': 339,
                'total_tokens': 347,
                'input_token_details': {'cache_read': 0}
            }
        ),
        'explanation': AIMessage(
            content='The joke generator asks for details about the house to craft a specific, funny joke. It 
provides examples and generic jokes while waiting for more information.',
            additional_kwargs={},
            response_metadata={
                'finish_reason': 'STOP',
                'model_name': 'gemini-2.5-flash-lite',
                'safety_ratings': [],
                'model_provider': 'google_genai'
            },
            id='lc_run--019e0364-1044-7303-876d-d8aba1ffb08b-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 539,
                'output_tokens': 29,
                'total_tokens': 568,
                'input_token_details': {'cache_read': 0}
            }
        )
    },
    (),
    {
        'configurable': {
            'thread_id': '2',
            'checkpoint_ns': '',
            'checkpoint_id': '1f14a368-b39a-620d-8002-9502050268ba'
        }
    },
    {'source': 'loop', 'step': 2, 'parents': {}},
    '2026-05-07T17:02:34.969537+00:00',
    {
        'configurable': {
            'thread_id': '2',
            'checkpoint_ns': '',
            'checkpoint_id': '1f14a368-83c7-6e28-8001-9d6d40540bb0'
        }
    },
    (),
    ()
]